In [5]:
import glob
import os
import numpy as np
import rasterio
from tqdm import tqdm

source_folder = "G:/Gedeelde drives/PRJ_SlimmeBeeldverwerking/PRJ_2026_SlimmeBeeldverwerking/Slimmebeeldverwerking/Bossenkansenkaart/Boshabitat_covariabelen/blocks/covariates_VAE_dataset"
output_folder = "E:/covariates_VAE_npz"
os.makedirs(output_folder, exist_ok=True)

target_size = (512, 512)
tif_files = glob.glob(os.path.join(source_folder, "*.tif"))

print("Pre-processing TIFF files to standard NPZ format...")
for file_path in tqdm(tif_files):
  filename = os.path.splitext(os.path.basename(file_path))[0]
  out_path = os.path.join(output_folder, f"{filename}.npz")

  # Overwrite broken zip files
  with rasterio.open(file_path) as src:
    data = src.read().astype(np.float32)

  current_channels, current_height, current_width = data.shape
  pad_h = target_size[0] - current_height
  pad_w = target_size[1] - current_width

  if pad_h > 0 or pad_w > 0:
    data = np.pad(
        data,
        ((0, 0), (0, pad_h), (0, pad_w)),
        mode="constant",
        constant_values=0,
    )

  mask = (~np.isnan(data)).astype(np.float32)
  data = np.nan_to_num(data, nan=0.0)

  # Use standard np.savez instead of np.savez_compressed
  np.savez(out_path, x=data, mask=mask)

Pre-processing TIFF files to standard NPZ format...


100%|██████████| 976/976 [53:23<00:00,  3.28s/it]  
